<font size=10>**RESULTS & FIGURES**</font> <a class="anchor" id='title'></a>

**Master's in Data Science & Advanced Analytics - NOVA IMS (26/27)**

**Course**: *Data Stream Mining — Practical Assignment I (PA48)*

**Project**: *Weather Windows for Outdoor Races — Angra do Heroísmo vs Braga*

**Group PA48**
- Beatriz Marques 20231605
- Duarte Fernandes 20231619

**Question**: *Which city offers more reliable weather windows for recurring outdoor races?*

**Pipeline step**: This notebook reproduces **step 4 (figures)** of `python src/practical_assignment.py --config config.yaml` from the saved audit outputs. It also provides results tables, consistency checks and unfilled spaces for the group’s original analyses; it does not retrain models.

| Notebook | What it does |
|---|---|
| 01 — `01_data_collection_and_quality.ipynb` | Data collection and quality |
| 02 — `02_target_and_features.ipynb` | Target, features and shared mask |
| 03 — `03_prequential_evaluation.ipynb` | Evaluation and audit outputs |
| **04 — `04_results_and_figures.ipynb`** | **Results, figures and audit checks** |

<font color='#BFD72F' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a>

- [1. Imports](#1)

- [2. Load Audit Outputs](#2)

- [3. Results Tables](#3)
    - [3.1 Mandatory Models — Macro-F1 and Recall](#3_1)
        - [Insights](#insights_1)
    - [3.2 Mandatory Models vs Baselines](#3_2)
    - [3.3 Computational Cost](#3_3)
        - [Insights](#insights_2)
    - [3.4 Model Complexity](#3_4)

- [4. Figures](#4)
    - [4.1 Cumulative Macro-F1](#4_1)
    - [4.2 Window Macro-F1](#4_2)
        - [Insights](#insights_3)
    - [4.3 Cumulative vs Window Recall](#4_3)
    - [4.4 Confusion Matrices](#4_4)
    - [4.5 Model Comparison](#4_5)
        - [Insights](#insights_4)
    - [4.6 Processing Cost](#4_6)
    - [4.7 Time-Performance Trade-off](#4_7)
        - [Insights](#insights_5)

- [5. Audit Checks](#5)

- [6. Original Analyses](#6)
    - [6.1 Stream Summary (both cities)](#6_1)
    - [6.2 O1 & O2 — Race-window Reliability and Spell Persistence](#6_2)
        - [Insights](#insights_6)
    - [6.3 O3 — Hour-of-day and Month Profiles by Event Source](#6_3)
        - [Insights](#insights_7)
    - [6.4 O4 — Effect of Temporal Context vs Cost](#6_4)
        - [Insights](#insights_8)

# <font color='#BFD72F' size=6>**1. Imports**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)

In [1]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

In [2]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

# Get the absolute path of the source code folder
source_code_path = os.path.abspath('../src')

# Add the source code folder to sys.path
if source_code_path not in sys.path:
    sys.path.append(source_code_path)

from data_loader import ROOT, load_config
from plots import generate_plots
from stream_analysis import plot_model_evidence
%matplotlib inline

# <font color='#BFD72F' size=6>**2. Load Audit Outputs**</font> <a class="anchor" id="2"></a>

[Back to TOC](#toc)

In [ ]:
cfg = load_config(ROOT / 'config.yaml')
out = ROOT / cfg['paths']['results']
for filename in ('metrics.csv', 'history.csv.gz', 'configurations.csv', 'stream_summary.csv'):
    if not (out / filename).exists():
        raise FileNotFoundError(f'Missing {out / filename}; run notebook 03 first.')
metrics = pd.read_csv(out / 'metrics.csv')
history = pd.read_csv(out / 'history.csv.gz', parse_dates=['timestamp'])
configurations = pd.read_csv(out / 'configurations.csv', index_col=0)
stream_summary = pd.read_csv(out / 'stream_summary.csv', index_col=0)
display(configurations)
print(f'Metrics: {len(metrics)} rows | History: {len(history)} rows')

FileNotFoundError: Required audit outputs are missing:
 - D:\OneDrive - NOVAIMS\S1\Data Stream Mining\Projects\Assignment 1\real-time-weather-data\results\stream_summary.csv
Run notebook 03 first, then rerun this cell.

# <font color='#BFD72F' size=6>**3. Results Tables**</font> <a class="anchor" id="3"></a>

[Back to TOC](#toc)

## <font color='#BFD72F' size=5>3.1 Mandatory Models — Macro-F1 and Recall</font> <a class="anchor" id="3_1"></a>

[Back to TOC](#toc)

In [ ]:
mandatory = metrics.loc[metrics['role'].eq('mandatory')].copy()
for metric in ('macro_f1', 'recall'):
    print(metric)
    display(mandatory.pivot(
        index='model', columns=['city', 'representation'], values=metric,
    ))

## <font color='#BFD72F' size=5>3.2 Mandatory Models vs Baselines</font> <a class="anchor" id="3_2"></a>

[Back to TOC](#toc)

In [ ]:
display(metrics[[
    'city', 'representation', 'model', 'role', 'accuracy', 'macro_f1', 'precision', 'recall',
]].sort_values(['city', 'representation', 'role', 'model']).reset_index(drop=True))

## <font color='#BFD72F' size=5>3.3 Computational Cost</font> <a class="anchor" id="3_3"></a>

[Back to TOC](#toc)

In [ ]:
cost = metrics.assign(
    pred_plus_update_ms_per_event=metrics['pred_ms_per_event'] + metrics['update_ms_per_event'],
)
for metric in ('pred_plus_update_ms_per_event', 'events_per_second'):
    print(metric)
    display(cost.pivot(index='model', columns=['city', 'representation'], values=metric))

## <font color='#BFD72F' size=5>3.4 Model Complexity</font> <a class="anchor" id="3_4"></a>

[Back to TOC](#toc)

In [ ]:
tree_models = ['HoeffdingTree', 'OnlineBagging', 'ARF']
complexity_columns = [
    'n_members', 'total_tree_nodes', 'mean_tree_height', 'drifts_detected', 'warnings_detected',
]
display(metrics.loc[metrics['model'].isin(tree_models),
    ['city', 'representation', 'model', 'role'] + complexity_columns,
].sort_values(['city', 'representation', 'model']).reset_index(drop=True))

# <font color='#BFD72F' size=6>**4. Figures**</font> <a class="anchor" id="4"></a>

[Back to TOC](#toc)

In [ ]:
figures_dir = out / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)
generate_plots(history, metrics, figures_dir)
figure_files = [
    'perf_cumulative_macro_f1.png',
    'perf_window_macro_f1.png',
    'perf_cumulative_vs_window_recall.png',
    'confusion_matrices.png',
    'model_comparison.png',
    'processing_cost.png',
    'time_performance_tradeoff.png',
]
plot_model_evidence(metrics, figures_dir / 'model_evidence.png')
# Stream profiles and city comparison are written by practical_assignment.py (stream_analysis.py)
figure_files += ['city_a_stream_profile.png', 'city_b_stream_profile.png',
                 'city_comparison.png', 'model_evidence.png']


## <font color='#BFD72F' size=5>4.1 Cumulative Macro-F1</font> <a class="anchor" id="4_1"></a>

[Back to TOC](#toc)

In [ ]:
display(Image(filename=str(figures_dir / 'perf_cumulative_macro_f1.png'), width=1000))

## <font color='#BFD72F' size=5>4.2 Window Macro-F1</font> <a class="anchor" id="4_2"></a>

[Back to TOC](#toc)

In [ ]:
display(Image(filename=str(figures_dir / 'perf_window_macro_f1.png'), width=1000))

## <font color='#BFD72F' size=5>4.3 Cumulative vs Window Recall</font> <a class="anchor" id="4_3"></a>

[Back to TOC](#toc)

In [ ]:
display(Image(filename=str(figures_dir / 'perf_cumulative_vs_window_recall.png'), width=1000))

## <font color='#BFD72F' size=5>4.4 Confusion Matrices</font> <a class="anchor" id="4_4"></a>

[Back to TOC](#toc)

In [ ]:
display(Image(filename=str(figures_dir / 'confusion_matrices.png'), width=1000))

## <font color='#BFD72F' size=5>4.5 Model Comparison</font> <a class="anchor" id="4_5"></a>

[Back to TOC](#toc)

In [ ]:
display(Image(filename=str(figures_dir / 'model_comparison.png'), width=1000))

## <font color='#BFD72F' size=5>4.6 Processing Cost</font> <a class="anchor" id="4_6"></a>

[Back to TOC](#toc)

In [ ]:
display(Image(filename=str(figures_dir / 'processing_cost.png'), width=1000))

## <font color='#BFD72F' size=5>4.7 Time-Performance Trade-off</font> <a class="anchor" id="4_7"></a>

[Back to TOC](#toc)

In [ ]:
display(Image(filename=str(figures_dir / 'time_performance_tradeoff.png'), width=1000))

# <font color='#BFD72F' size=6>**5. Audit Checks**</font> <a class="anchor" id="5"></a>

[Back to TOC](#toc)

In [ ]:
saved_include_baselines = str(configurations.loc['include_baselines', 'value']).strip().lower()
assert saved_include_baselines in ('true', 'false'), 'Invalid saved baseline switch.'
include_baselines = saved_include_baselines == 'true'
assert include_baselines == cfg['baselines']['include'], (
    'The saved baseline setting differs from config.yaml; use the matching configuration.'
)
expected_rows = 32 if include_baselines else 20
assert len(metrics) == expected_rows, f'Expected {expected_rows} metrics rows, got {len(metrics)}.'
assert metrics['n_eval'].notna().all(), 'Missing n_eval values.'
assert metrics.groupby('city')['n_eval'].nunique().eq(1).all(), (
    'n_eval must be identical across every representation/model run within each city.'
)
for filename in figure_files:
    assert (figures_dir / filename).is_file(), f'Missing figure: {filename}'
print(f"Frozen thresholds: {configurations.loc['thresholds', 'value']}")
print('All audit checks passed')

# <font color='#BFD72F' size=6>**6. Original Analyses**</font> <a class="anchor" id="6"></a>

[Back to TOC](#toc)

**Guidelines 3.2 requires ≥2 original analyses/visualisations beyond the pipeline figures.** The analyses below are designed for the PA48 business question and are produced by `src/stream_analysis.py` (run automatically by `practical_assignment.py`):

| ID | Analysis | Figure | Business / technical question it serves |
|---|---|---|---|
| **O1** | Race-window reliability: share of days whose local **08:00–12:00** race window has no adverse hour, by month | `city_comparison.png` (a) | How many race mornings are usable in each city, and when? |
| **O2** | Adverse / clear spell durations and persistence $P(y_t=1\mid y_{t-1}=1)$ | `city_comparison.png` (b), `city_*_stream_profile.png` (d) | How long do adverse periods last once they start? |
| **O3** | Local hour-of-day and calendar-month event rate split by source (rain only / wind only / both) | `city_*_stream_profile.png` (b, c) | When and why do adverse hours happen? |
| **O4** | Change in Macro-F1 and event recall from Rep A to Rep B next to the cost multiplier | `model_evidence.png` | Is temporal context worth its extra cost? |

The race window (start hour and length) is set in `config.yaml → analysis.race_window`. Local hours use `Atlantic/Azores` and `Europe/Lisbon`; the stream itself stays in UTC.

## <font color='#BFD72F' size=5>6.1 Stream Summary (both cities)</font> <a class="anchor" id="6_1"></a>

[Back to TOC](#toc)

Display **`stream_summary.csv`**: the same descriptive, event, persistence and window statistics computed with identical definitions for both cities. Rates are shares of hours (or days for race windows).

In [ ]:
display(stream_summary)

## <font color='#BFD72F' size=5>6.2 O1 & O2 — Race-window Reliability and Spell Persistence</font> <a class="anchor" id="6_2"></a>

[Back to TOC](#toc)

In [ ]:
display(Image(filename=str(figures_dir / 'city_comparison.png'), width=1000))

## <font color='#BFD72F' size=5>6.3 O3 — Hour-of-day and Month Profiles by Event Source</font> <a class="anchor" id="6_3"></a>

[Back to TOC](#toc)

In [ ]:
for key in ('city_a', 'city_b'):
    display(Image(filename=str(figures_dir / f'{key}_stream_profile.png'), width=1000))

## <font color='#BFD72F' size=5>6.4 O4 — Effect of Temporal Context vs Cost</font> <a class="anchor" id="6_4"></a>

[Back to TOC](#toc)

In [ ]:
display(Image(filename=str(figures_dir / 'model_evidence.png'), width=1000))
mand = metrics[metrics.role == 'mandatory'].assign(cost_ms=lambda d: d.pred_ms_per_event + d.update_ms_per_event)
piv = mand.pivot_table(index=['city', 'model'], columns='representation', values=['macro_f1', 'recall', 'cost_ms'])
display(pd.DataFrame({
    'delta_macro_f1': piv['macro_f1']['B'] - piv['macro_f1']['A'],
    'delta_recall': piv['recall']['B'] - piv['recall']['A'],
    'cost_ratio_B_over_A': piv['cost_ms']['B'] / piv['cost_ms']['A'],
}).round(4))